In [ ]:
# ============================================================
# PLOT CLUSTERED STORES ON AN INTERACTIVE MAP
# Works in: Google Colab, Jupyter Notebook, or local Python
#
# HOW TO USE:
#   Option A (recommended): Upload 'location_with_clusters.xlsx'
#                           (already has cluster columns from previous step)
#   Option B:               Upload 'solution_deliveries_Monday.xlsx'
#                           (will run clustering automatically)
#
# In Google Colab: the script will prompt you to upload the file
# In Jupyter/local: place the .xlsx in the same folder as this script
#
# Outputs:
#   cluster_map_interactive.html  — zoomable, click-for-info
#   cluster_map_static.png        — high-quality static PNG
# ============================================================

import os, glob, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.spatial import ConvexHull

# ── AUTO-INSTALL FOLIUM IF MISSING ───────────────────────────
try:
    import folium
    from folium.plugins import MiniMap
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"])
    import folium
    from folium.plugins import MiniMap

# ══════════════════════════════════════════════════════════════
# ── SETTINGS ─────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
EXCEL_FILE   = None          # None = auto-detect / prompt upload
CLUSTER_COL  = "Cluster_KMedoids"   # or "Cluster_Hierarchical"
DC_CLUSTER   = 0
MAP_HTML_OUT = "cluster_map_interactive.html"
MAP_PNG_OUT  = "cluster_map_static.png"

CLUSTER_COLOURS = {
    1: "#E63946", 2: "#2196F3", 3: "#FF9800", 4: "#4CAF50",
    5: "#9C27B0", 6: "#00BCD4", 7: "#FF5722", 8: "#607D8B",
    9: "#8BC34A", 10:"#F06292",
}
DC_COLOUR = "#1A1A2E"

# ══════════════════════════════════════════════════════════════
# ── 1) FIND / UPLOAD THE EXCEL FILE ──────────────────────────
# ══════════════════════════════════════════════════════════════

def find_excel():
    # 1. Manual override
    if EXCEL_FILE and os.path.exists(EXCEL_FILE):
        return EXCEL_FILE

    # 2. Known filenames in current working directory
    for name in ["location_with_clusters.xlsx", "solution_deliveries_Monday.xlsx"]:
        if os.path.exists(name):
            return name

    # 3. Any .xlsx in cwd
    found = glob.glob("*.xlsx")
    if found:
        print(f"Auto-detected Excel file: {found[0]}")
        return found[0]

    # 4. Colab /content folder
    colab_files = glob.glob("/content/*.xlsx")
    if colab_files:
        return colab_files[0]

    # 5. Colab upload prompt
    try:
        from google.colab import files
        print("Please upload your Excel file (location_with_clusters.xlsx or solution_deliveries_Monday.xlsx):")
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            print(f"Uploaded: {fname}")
            return fname
    except (ImportError, Exception):
        pass

    # 6. Manual path
    path = input("Enter full path to your Excel file: ").strip().strip('"').strip("'")
    if os.path.exists(path):
        return path

    raise FileNotFoundError(
        "\n\nCould not find Excel file. Please do ONE of the following:\n"
        "  a) Place 'location_with_clusters.xlsx' in the same folder as this script\n"
        "  b) Set  EXCEL_FILE = '/full/path/to/your_file.xlsx'  near the top of the script\n"
        "  c) Run in Google Colab — the upload dialog will appear automatically\n"
    )

excel_path = find_excel()
print(f"Using file: {excel_path}")

# ══════════════════════════════════════════════════════════════
# ── 2) LOAD DATA  ─────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════

xl = pd.ExcelFile(excel_path)
print(f"Sheets found: {xl.sheet_names}")

# ── Case A: location_with_clusters.xlsx (already has Cluster columns) ──
if "LocationTable_Clustered" in xl.sheet_names:
    df = pd.read_excel(excel_path, sheet_name="LocationTable_Clustered")
    print("Loaded sheet: LocationTable_Clustered")

# ── Case B: solution_deliveries_Monday.xlsx (raw data — cluster on the fly) ──
elif "LocationTable" in xl.sheet_names:
    print("Raw file detected — running K-Medoids clustering on the fly...")

    loc_df = pd.read_excel(excel_path, sheet_name="LocationTable")
    loc_df = loc_df.dropna(subset=["ZIP"]).copy()
    loc_df["ZIP"]   = loc_df["ZIP"].astype(float).astype(int).astype(str).str.zfill(5)
    loc_df["ZIPID"] = loc_df["ZIPID"].astype(float).astype(int)
    loc_df = loc_df[loc_df["ZIP"].str.match(r"^\d{5}$")].reset_index(drop=True)
    id_to_zip = dict(zip(loc_df["ZIPID"], loc_df["ZIP"]))

    dist_raw = pd.read_excel(excel_path, sheet_name="distance", header=1)
    dist_raw["Zip"] = dist_raw["Zip"].astype(float).astype(int).astype(str).str.zfill(5)
    dist_raw = dist_raw.set_index("Zip").drop(columns=["id"], errors="ignore")
    dist_raw.columns = [id_to_zip.get(int(float(str(c))), str(c)) for c in dist_raw.columns]
    dist_raw = dist_raw.apply(pd.to_numeric, errors="coerce")

    DC_ZIP       = "01887"
    common_zips  = [z for z in loc_df["ZIP"] if z in dist_raw.index and z in dist_raw.columns]
    dist_sq      = dist_raw.loc[common_zips, common_zips].values.astype(float)
    np.fill_diagonal(dist_sq, 0)
    dist_sq      = (dist_sq + dist_sq.T) / 2
    delivery_z   = [z for z in common_zips if z != DC_ZIP]
    d_idxs       = [common_zips.index(z) for z in delivery_z]
    dist_del     = dist_sq[np.ix_(d_idxs, d_idxs)]

    def kmedoids(D, k, n_init=10, seed=42):
        rng = np.random.default_rng(seed)
        n   = D.shape[0]
        best_cost, best_labels = float("inf"), None
        for _ in range(n_init):
            meds = list(rng.choice(n, k, replace=False))
            for _ in range(300):
                labels = np.argmin(D[:, meds], axis=1)
                new_meds = []
                for ci in range(k):
                    mem = np.where(labels == ci)[0]
                    new_meds.append(meds[ci] if len(mem)==0
                                    else mem[np.argmin(D[np.ix_(mem,mem)].sum(axis=1))])
                if sorted(new_meds)==sorted(meds): break
                meds = new_meds
            labels = np.argmin(D[:, meds], axis=1)
            cost = sum(D[i, meds[labels[i]]] for i in range(n))
            if cost < best_cost:
                best_cost, best_labels = cost, labels.copy()+1
        return best_labels

    N = 8
    labels = kmedoids(dist_del, k=N)
    loc_df["Cluster_KMedoids"] = loc_df["ZIP"].map(dict(zip(delivery_z, labels)))
    if DC_ZIP in loc_df["ZIP"].values:
        loc_df.loc[loc_df["ZIP"]==DC_ZIP, "Cluster_KMedoids"] = 0
    loc_df["Cluster_Hierarchical"] = loc_df["Cluster_KMedoids"]   # same for simplicity
    df = loc_df
    CLUSTER_COL = "Cluster_KMedoids"
    print(f"Clustering complete — {N} clusters created")

else:
    raise ValueError(
        f"Could not find 'LocationTable_Clustered' or 'LocationTable' sheet.\n"
        f"Sheets found: {xl.sheet_names}"
    )

# ── Normalise dataframe ───────────────────────────────────────
df["ZIP"] = df["ZIP"].astype(str).str.zfill(5)
df = df.dropna(subset=["X","Y"]).copy()

if CLUSTER_COL not in df.columns:
    alt = next((c for c in ["Cluster_Hierarchical","Cluster_KMedoids"] if c in df.columns), None)
    if alt is None:
        raise ValueError(f"No cluster column found. Columns: {df.columns.tolist()}")
    print(f"Column '{CLUSTER_COL}' not found — using '{alt}'")
    CLUSTER_COL = alt

df[CLUSTER_COL] = pd.to_numeric(df[CLUSTER_COL], errors="coerce")
df = df.dropna(subset=[CLUSTER_COL]).copy()
df[CLUSTER_COL] = df[CLUSTER_COL].astype(int)

delivery_df = df[df[CLUSTER_COL] != DC_CLUSTER]
dc_df       = df[df[CLUSTER_COL] == DC_CLUSTER]
clusters    = sorted(delivery_df[CLUSTER_COL].unique())

print(f"\n{'='*45}")
print(f"  Stores loaded : {len(df)}")
print(f"  Cluster col   : {CLUSTER_COL}")
print(f"  Clusters      : {len(clusters)}")
for ci in clusters:
    sub = delivery_df[delivery_df[CLUSTER_COL]==ci]
    print(f"    Cluster {ci:2d}: {len(sub):3d} stores")
print(f"{'='*45}\n")

# ══════════════════════════════════════════════════════════════
# ── 3) INTERACTIVE FOLIUM MAP ─────────────────────────────────
# ══════════════════════════════════════════════════════════════

m = folium.Map(
    location=[df["Y"].mean(), df["X"].mean()],
    zoom_start=8, tiles="CartoDB positron", control_scale=True,
)
folium.TileLayer("CartoDB dark_matter", name="Dark Mode").add_to(m)
folium.TileLayer("OpenStreetMap",       name="Street Map").add_to(m)

# Convex hull regions
hull_group = folium.FeatureGroup(name="Cluster Regions", show=True)
for ci in clusters:
    sub    = delivery_df[delivery_df[CLUSTER_COL]==ci]
    colour = CLUSTER_COLOURS.get(ci, "#999")
    pts    = sub[["Y","X"]].values
    if len(pts) >= 3:
        try:
            h = ConvexHull(pts)
            pts_poly = pts[h.vertices].tolist() + [pts[h.vertices[0]].tolist()]
            folium.Polygon(locations=pts_poly, color=colour, weight=1.5,
                           fill=True, fill_color=colour, fill_opacity=0.10,
                           tooltip=f"Cluster {ci}").add_to(hull_group)
        except Exception:
            pass
hull_group.add_to(m)

# Store markers
for ci in clusters:
    sub    = delivery_df[delivery_df[CLUSTER_COL]==ci]
    colour = CLUSTER_COLOURS.get(ci, "#999")
    grp    = folium.FeatureGroup(name=f"Cluster {ci}  ({len(sub)} stores)", show=True)
    for _, row in sub.iterrows():
        city, state = str(row.get("CITY","")), str(row.get("STATE",""))
        popup_html = (
            f'<div style="font-family:Segoe UI,sans-serif;min-width:155px">'
            f'<div style="background:{colour};color:#fff;padding:5px 9px;'
            f'border-radius:4px 4px 0 0;font-weight:700">Cluster {ci}</div>'
            f'<div style="padding:7px 9px;background:#fafafa;border:1px solid #eee;'
            f'border-top:none;border-radius:0 0 4px 4px">'
            f'<b>📍 {city}, {state}</b><br>'
            f'<span style="color:#666;font-size:11px">ZIP: {row["ZIP"]}</span><br>'
            f'<span style="color:#666;font-size:11px">Lat {row["Y"]:.4f} | Lon {row["X"]:.4f}</span>'
            f'</div></div>'
        )
        folium.CircleMarker(
            location=[row["Y"], row["X"]], radius=7,
            color="white", weight=1.5,
            fill=True, fill_color=colour, fill_opacity=0.92,
            tooltip=f"{city} ({row['ZIP']}) — Cluster {ci}",
            popup=folium.Popup(popup_html, max_width=210),
        ).add_to(grp)
    grp.add_to(m)

# DC marker
dc_grp = folium.FeatureGroup(name="Distribution Centre (DC)", show=True)
for _, row in dc_df.iterrows():
    city = str(row.get("CITY",""))
    popup_html = (
        f'<div style="font-family:Segoe UI,sans-serif;min-width:165px">'
        f'<div style="background:{DC_COLOUR};color:#fff;padding:5px 9px;'
        f'border-radius:4px 4px 0 0;font-weight:700">🏭 Distribution Centre</div>'
        f'<div style="padding:7px 9px;background:#fafafa;border:1px solid #eee;'
        f'border-top:none;border-radius:0 0 4px 4px">'
        f'<b>{city}</b><br>'
        f'<span style="color:#666;font-size:11px">ZIP: {row["ZIP"]}</span>'
        f'</div></div>'
    )
    folium.Marker(
        location=[row["Y"], row["X"]],
        popup=folium.Popup(popup_html, max_width=210),
        tooltip=f"DC — {city} ({row['ZIP']})",
        icon=folium.Icon(color="black", icon="home", prefix="fa"),
    ).add_to(dc_grp)
dc_grp.add_to(m)

# Legend HTML overlay
legend_items = "".join(
    f'<div style="display:flex;align-items:center;gap:7px;margin:3px 0">'
    f'<div style="width:12px;height:12px;border-radius:50%;'
    f'background:{CLUSTER_COLOURS.get(ci,"#999")};border:1.5px solid #fff;'
    f'box-shadow:0 0 3px rgba(0,0,0,.25)"></div>'
    f'<span style="font-size:11px;color:#333">Cluster {ci} '
    f'<span style="color:#888">({len(delivery_df[delivery_df[CLUSTER_COL]==ci])} stores)</span>'
    f'</span></div>'
    for ci in clusters
) + (
    f'<div style="display:flex;align-items:center;gap:7px;margin:6px 0 0;'
    f'border-top:1px solid #e0e0e0;padding-top:5px">'
    f'<div style="width:12px;height:12px;background:{DC_COLOUR};border-radius:3px"></div>'
    f'<span style="font-size:11px;color:#333">Distribution Centre</span></div>'
)
m.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:30px;left:15px;z-index:9999;background:#fff;'
    f'padding:11px 14px;border-radius:8px;box-shadow:0 2px 12px rgba(0,0,0,.18);'
    f'font-family:Segoe UI,sans-serif;max-width:195px">'
    f'<div style="font-weight:700;font-size:11px;color:#111;margin-bottom:7px;'
    f'text-transform:uppercase;letter-spacing:.5px">🗺 Cluster Legend</div>'
    f'{legend_items}</div>'
))
m.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:9999;'
    'background:rgba(255,255,255,.95);padding:7px 18px;border-radius:6px;'
    'box-shadow:0 2px 10px rgba(0,0,0,.15);font-family:Segoe UI,sans-serif;'
    'font-size:13px;font-weight:700;color:#1a1a2e;white-space:nowrap">'
    '📦 Monday Delivery Store Clusters</div>'
))

folium.LayerControl(collapsed=False).add_to(m)
MiniMap(toggle_display=True, tile_layer="CartoDB positron").add_to(m)
m.save(MAP_HTML_OUT)
print(f"Interactive map saved: {MAP_HTML_OUT}")

# Inline display in Jupyter/Colab
try:
    from IPython.display import display, IFrame
    display(IFrame(MAP_HTML_OUT, width="100%", height="540px"))
except Exception:
    pass

# ══════════════════════════════════════════════════════════════
# ── 4) STATIC PNG MAP ─────────────────────────────────────────
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(16,12), facecolor="#1A1A2E")
ax.set_facecolor("#1A1A2E")
ax.grid(True, linestyle="--", alpha=0.07, color="white", linewidth=0.5)

for ci in clusters:
    sub    = delivery_df[delivery_df[CLUSTER_COL]==ci]
    colour = CLUSTER_COLOURS.get(ci,"#999")
    pts    = sub[["X","Y"]].values
    if len(pts) >= 3:
        try:
            h = ConvexHull(pts)
            ax.add_patch(plt.Polygon(pts[h.vertices], closed=True,
                facecolor=colour, alpha=0.09,
                edgecolor=colour, linewidth=0.9, linestyle="--"))
        except Exception:
            pass
    ax.scatter(sub["X"], sub["Y"], c=colour, s=72, zorder=4,
               edgecolors="white", linewidths=0.6, alpha=0.95,
               label=f"Cluster {ci}  ({len(sub)})")
    for _, row in sub.iterrows():
        ax.annotate(str(ci),(row["X"],row["Y"]), fontsize=4.5,
                    ha="center", va="center", color="white",
                    fontweight="bold", zorder=5)

if not dc_df.empty:
    ax.scatter(dc_df["X"], dc_df["Y"], c=DC_COLOUR, s=280, marker="D",
               zorder=6, edgecolors="#00E5FF", linewidths=2.0,
               label="Distribution Centre")
    for _, row in dc_df.iterrows():
        ax.annotate("DC",(row["X"],row["Y"]), fontsize=6,
                    ha="center", va="center", color="#00E5FF",
                    fontweight="bold", zorder=7)

top_cities = delivery_df.groupby("CITY").size().nlargest(20).index.tolist()
for _, row in df[df["CITY"].isin(top_cities)].iterrows():
    ax.annotate(row["CITY"],(row["X"],row["Y"]),
                xytext=(3,5), textcoords="offset points",
                fontsize=5.5, color="#CCCCCC", alpha=0.85,
                path_effects=[pe.withStroke(linewidth=1.5, foreground="#1A1A2E")])

ax.set_xlabel("Longitude", color="#AAAAAA", fontsize=10, labelpad=8)
ax.set_ylabel("Latitude",  color="#AAAAAA", fontsize=10, labelpad=8)
ax.tick_params(colors="#666", labelsize=8)
for spine in ax.spines.values(): spine.set_edgecolor("#333355")
ax.set_title(
    f"Monday Delivery Store Clusters  |  {CLUSTER_COL.replace('Cluster_','')}  |  {len(clusters)} clusters",
    color="white", fontsize=14, fontweight="bold", pad=14, fontfamily="monospace"
)
legend = ax.legend(loc="lower right", fontsize=8, framealpha=0.15,
                   facecolor="#0D0D1A", edgecolor="#444466",
                   labelcolor="white", markerscale=1.1,
                   title="Cluster  (stores)", title_fontsize=8)
legend.get_title().set_color("#AAAAAA")
ax.text(0.01, 0.98,
        f"Stores: {len(delivery_df)}  |  Clusters: {len(clusters)}  |  Method: {CLUSTER_COL.replace('Cluster_','')}",
        transform=ax.transAxes, va="top", ha="left", fontsize=8,
        color="#AAAAAA", fontfamily="monospace",
        bbox=dict(facecolor="#0D0D1A", alpha=0.6, edgecolor="#333355", boxstyle="round,pad=0.4"))

plt.tight_layout()
plt.savefig(MAP_PNG_OUT, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"Static map saved: {MAP_PNG_OUT}")

# Download in Colab
try:
    from google.colab import files
    files.download(MAP_HTML_OUT)
    files.download(MAP_PNG_OUT)
    print("Files sent to browser downloads.")
except Exception:
    pass

print("\n✅ Done!")

Using file: location_with_clusters.xlsx
Sheets found: ['LocationTable_Clustered', 'Summary_Hierarchical', 'Summary_KMedoids']
Loaded sheet: LocationTable_Clustered

  Stores loaded : 123
  Cluster col   : Cluster_KMedoids
  Clusters      : 8
    Cluster  1:  18 stores
    Cluster  2:  10 stores
    Cluster  3:   5 stores
    Cluster  4:  31 stores
    Cluster  5:  16 stores
    Cluster  6:   9 stores
    Cluster  7:  24 stores
    Cluster  8:   9 stores

Interactive map saved: cluster_map_interactive.html


Static map saved: cluster_map_static.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files sent to browser downloads.

✅ Done!


**Check Scheduling** Monday

In [ ]:
!pip -q install ortools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 16.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.


In [ ]:
# # ============================================================
# # COMPLETE, WORKING END-TO-END CODE (Colab / Python) — MONDAY
# # + Soft "cluster grouping" preference using LocationTable!Cluster_Hierarchical
# # ============================================================

# import glob
# import math
# import pandas as pd
# from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# # ==================== SETTINGS ====================
# DC = "01887"
# CAPACITY   = 3200         # cubic feet per trailer

# OPEN       = 8  * 60     # 08:00 in minutes
# CLOSE      = 18 * 60     # 18:00 in minutes

# # HOS limits (minutes)
# MAX_DRIVE  = 11 * 60     # 660 min — cumulative DRIVING only
# MAX_DUTY   = 14 * 60     # 840 min — cumulative ON-DUTY (drive + service + wait)
# REST_MIN   = 10 * 60     # 600 min — mandatory rest before reset

# SOLVER_SECONDS = 120     # OR-Tools time limit
# EXTRA_VEHICLES = 30      # padding so solver has enough vehicles

# DAY = 24 * 60            # minutes in a day (for display)

# # ---------- Cluster preference strength ----------
# # Penalty added (in "miles") when consecutive stops are in different clusters.
# # Larger => stronger grouping, but too large may increase total miles a lot.
# CLUSTER_MISMATCH_PENALTY_MILES = 30.0  # try 10, 20, 30, 50 if needed


# # ==================== HELPERS ====================

# def pick_sheet(candidates, contains_all=()):
#     keys = [k.lower() for k in contains_all]
#     for s in candidates:
#         if all(k in s.lower() for k in keys):
#             return s
#     return None

# def fmt_day_time(total_min):
#     m = float(total_min)
#     d = int(m // DAY)
#     tod = m - d * DAY
#     hh, mm = divmod(int(round(tod)), 60)
#     if mm == 60:
#         hh += 1
#         mm = 0
#     return f"D{d} {hh:02d}:{mm:02d}"

# def fmt_hhmm(total_min):
#     m = float(total_min)
#     hh, mm = divmod(int(round(m)), 60)
#     if mm == 60:
#         hh += 1
#         mm = 0
#     return f"{hh:02d}:{mm:02d}"


# # ==================== 1) LOCATE EXCEL ====================

# xlsx_files = sorted(glob.glob("*.xlsx"))
# if not xlsx_files:
#     raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
# file_path = xlsx_files[0]
# print("Using file:", file_path)

# xl = pd.ExcelFile(file_path)
# print("Sheets:", xl.sheet_names)


# # ==================== 2) PICK SHEETS ====================

# distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

# monday_sheet = (
#     pick_sheet(xl.sheet_names, contains_all=("monday", "final"))
#     or pick_sheet(xl.sheet_names, contains_all=("monday",))
# )
# if monday_sheet is None:
#     raise ValueError("Could not find a Monday sheet (e.g., 'Monday Final').")

# # Location table sheet (where clusters live)
# location_sheet = (
#     pick_sheet(xl.sheet_names, contains_all=("location", "table"))
#     or pick_sheet(xl.sheet_names, contains_all=("location",))
# )
# if location_sheet is None:
#     raise ValueError("Could not find a LocationTable sheet for clusters.")

# print("Distance sheet:", distance_sheet)
# print("Monday sheet  :", monday_sheet)
# print("Location sheet:", location_sheet)


# # ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

# raw = pd.read_excel(file_path, sheet_name=distance_sheet)
# raw.columns = [str(c).strip() for c in raw.columns]
# raw = raw.rename(columns={raw.columns[0]: "Zip"})
# raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
# raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
# raw = raw.dropna(axis=1, how="all")

# raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
# raw = raw.set_index("Zip")
# raw.columns = (
#     pd.Series(raw.columns).astype(str).str.strip()
#       .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
# )

# dist_df = raw.apply(pd.to_numeric, errors="coerce")
# common  = dist_df.index.intersection(dist_df.columns)
# dist_df = dist_df.loc[common, common]
# print("Distance matrix shape:", dist_df.shape)

# if dist_df.isna().any().any():
#     bad = dist_df.isna().stack()
#     bad = bad[bad].index.tolist()[:15]
#     raise ValueError(f"Distance matrix has NaNs: {bad}")


# # ==================== 3.5) READ LOCATION TABLE (ZIP -> Cluster) ====================

# loc = pd.read_excel(file_path, sheet_name=location_sheet)
# loc.columns = [str(c).strip() for c in loc.columns]

# zip_loc_col = next((c for c in ["ZIP", "Zip", "zip", "StoreZIP", "Store Zip"] if c in loc.columns), None)
# clu_col = next((c for c in ["Cluster_Hierarchical", "cluster_hierarchical", "Cluster"] if c in loc.columns), None)

# if zip_loc_col is None or clu_col is None:
#     raise ValueError(f"Location sheet must have ZIP and Cluster_Hierarchical. Found columns: {loc.columns.tolist()}")

# loc[zip_loc_col] = loc[zip_loc_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
# loc[clu_col] = pd.to_numeric(loc[clu_col], errors="coerce")
# loc = loc.dropna(subset=[zip_loc_col, clu_col])

# zip_to_cluster = dict(zip(loc[zip_loc_col], loc[clu_col].astype(int)))
# print("Clusters loaded for ZIPs:", len(zip_to_cluster))


# # ==================== 4) READ & CLEAN MONDAY DATA ====================

# day_df = pd.read_excel(file_path, sheet_name=monday_sheet)

# zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
# cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)

# if zip_col is None or cube_col is None:
#     raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

# day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
# day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
# day_df = day_df[day_df[zip_col].notna()].copy()

# unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

# print(f"Monday stores (original rows): {len(day_df)}")
# print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")


# # ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

# split_rows = []
# for _, r in day_df.iterrows():
#     z    = r[zip_col]
#     cube = float(r[cube_col])
#     if cube <= 0:
#         continue

#     unload_val = None
#     if unload_col:
#         try:
#             unload_val = float(r[unload_col])
#         except Exception:
#             pass

#     k         = math.ceil(cube / CAPACITY)
#     remaining = cube
#     for part in range(1, k + 1):
#         chunk     = min(CAPACITY, remaining)
#         remaining -= chunk
#         split_rows.append({
#             "BaseZIP"        : z,
#             "VisitID"        : f"{z}#{part}" if k > 1 else z,
#             "Cube"           : chunk,
#             "UnloadFromSheet": unload_val,
#         })

# split_df = pd.DataFrame(split_rows)
# print(f"Total delivery stops after splitting: {len(split_df)}")


# # ==================== 6) BUILD NODE ARRAYS ====================

# nodes_id = [DC] + split_df["VisitID"].tolist()   # index 0 = depot
# base_zip = [DC] + split_df["BaseZIP"].tolist()

# missing = [z for z in set(base_zip) if z not in dist_df.index]
# if missing:
#     raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

# # Cluster per node (depot cluster = -1)
# node_cluster = [-1] + [zip_to_cluster.get(z, -1) for z in split_df["BaseZIP"].tolist()]
# missing_cluster = sorted({z for z in set(split_df["BaseZIP"]) if z not in zip_to_cluster})
# if missing_cluster:
#     print("WARNING: These ZIPs have no cluster in LocationTable (treated as -1):", missing_cluster[:20])

# n = len(base_zip)

# # Distance matrix (miles)
# dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]

# # Travel time (minutes): 40 mph → 1 mile = 1.5 min
# travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

# # Demand (cube) per node; depot = 0
# demands = [0] + split_df["Cube"].astype(float).tolist()

# # Service time per node
# service_times = [0.0]
# for _, r in split_df.iterrows():
#     if unload_col and pd.notna(r["UnloadFromSheet"]):
#         svc = max(0.0, float(r["UnloadFromSheet"]))
#     else:
#         svc = max(30.0, 0.030 * float(r["Cube"]))
#     service_times.append(svc)


# # ==================== 7) OR-TOOLS VRP (capacity + time window + cluster preference) ====================

# min_trailers  = math.ceil(sum(demands) / CAPACITY)
# num_vehicles  = min_trailers + EXTRA_VEHICLES
# print(f"Min trailers (capacity only): {min_trailers}")
# print(f"Vehicles given to OR-Tools  : {num_vehicles}")

# manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
# routing = pywrapcp.RoutingModel(manager)

# # --- ARC COST = distance + penalty(if cluster mismatch) ---
# cluster_penalty_cost = int(CLUSTER_MISMATCH_PENALTY_MILES * 1000)

# def arc_cost_cb(fi, ti):
#     i = manager.IndexToNode(fi)
#     j = manager.IndexToNode(ti)

#     base_cost = int(dist_miles[i][j] * 1000)

#     # apply penalty only when both are real stops (not depot)
#     if i != 0 and j != 0:
#         ci = node_cluster[i]
#         cj = node_cluster[j]
#         if ci != -1 and cj != -1 and ci != cj:
#             base_cost += cluster_penalty_cost

#     return base_cost

# routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(arc_cost_cb))

# def demand_cb(fi):
#     return int(round(demands[manager.IndexToNode(fi)]))

# routing.AddDimensionWithVehicleCapacity(
#     routing.RegisterUnaryTransitCallback(demand_cb),
#     0, [CAPACITY] * num_vehicles, True, "Capacity"
# )

# def time_cb(fi, ti):
#     i = manager.IndexToNode(fi)
#     j = manager.IndexToNode(ti)
#     travel = travel_min[i][j]
#     svc = service_times[i] if i != 0 else 0.0
#     return int(round(travel + svc))

# time_idx = routing.RegisterTransitCallback(time_cb)
# routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
# time_dim = routing.GetDimensionOrDie("Time")

# # Store windows: arrival must allow service to finish by 18:00
# for node in range(1, n):
#     latest = int(CLOSE - math.ceil(service_times[node]))
#     if latest < OPEN:
#         raise ValueError(
#             f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} min too large for 08–18."
#         )
#     idx = manager.NodeToIndex(node)
#     time_dim.CumulVar(idx).SetRange(OPEN, latest)

# # Depot starts in [0, 08:00]
# for v in range(num_vehicles):
#     time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

# params = pywrapcp.DefaultRoutingSearchParameters()
# params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
# params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
# params.time_limit.FromSeconds(SOLVER_SECONDS)

# solution = routing.SolveWithParameters(params)
# if not solution:
#     raise RuntimeError(
#         "OR-Tools found no feasible solution.\n"
#         "All stops cannot be served 08:00–18:00 with given capacity.\n"
#         "Try increasing EXTRA_VEHICLES or check input data."
#     )

# # Extract non-empty routes
# routes = []
# for v in range(num_vehicles):
#     idx = routing.Start(v)
#     nxt = solution.Value(routing.NextVar(idx))
#     if routing.IsEnd(nxt):
#         continue
#     route = []
#     while not routing.IsEnd(idx):
#         route.append(manager.IndexToNode(idx))
#         idx = solution.Value(routing.NextVar(idx))
#     route.append(0)
#     routes.append(route)

# print(f"Routes from OR-Tools (before HOS split): {len(routes)}")


# # ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

# def split_for_hos(route):
#     if len(route) <= 2:
#         return [route]

#     first = route[1]
#     dispatch = OPEN - travel_min[0][first]
#     t = dispatch
#     drive = 0.0
#     duty = 0.0
#     prev = 0

#     for k, cur in enumerate(route[1:-1], start=1):
#         leg_travel = travel_min[prev][cur]

#         if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
#             left  = route[:k] + [0]
#             right = [0] + route[k:]
#             return split_for_hos(left) + split_for_hos(right)

#         t += leg_travel
#         drive += leg_travel
#         duty  += leg_travel

#         if t < OPEN:
#             duty += (OPEN - t)
#             t = OPEN

#         svc = service_times[cur]
#         if t + svc > CLOSE + 1e-6:
#             left  = route[:k] + [0]
#             right = [0] + route[k:]
#             return split_for_hos(left) + split_for_hos(right)

#         duty += svc
#         t += svc

#         if drive > MAX_DRIVE or duty > MAX_DUTY:
#             left  = route[:k] + [0]
#             right = [0] + route[k:]
#             return split_for_hos(left) + split_for_hos(right)

#         prev = cur

#     return [route]

# hos_routes = []
# for r in routes:
#     hos_routes.extend(split_for_hos(r))

# routes = [r for r in hos_routes if len(r) > 2]
# print(f"Routes after HOS enforcement: {len(routes)}")


# # ==================== 9) BUILD SCHEDULE ====================

# schedule_rows = []
# summary_rows  = []
# truck_num     = 0

# for route in routes:
#     if len(route) <= 2:
#         continue

#     truck_num += 1
#     first = route[1]

#     dispatch = OPEN - travel_min[0][first]
#     t = dispatch
#     drive = 0.0
#     duty  = 0.0
#     miles = 0.0
#     prev  = 0
#     stop_n = 0

#     for cur in route[1:-1]:
#         leg_t  = travel_min[prev][cur]
#         leg_mi = dist_miles[prev][cur]

#         t += leg_t
#         drive += leg_t
#         duty  += leg_t
#         miles += leg_mi

#         if t < OPEN:
#             duty += (OPEN - t)
#             t = OPEN

#         arrival = t
#         svc = service_times[cur]
#         depart = arrival + svc

#         duty += svc
#         t = depart
#         stop_n += 1

#         schedule_rows.append({
#             "Truck"           : truck_num,
#             "Stop#"           : stop_n,
#             "VisitID"         : nodes_id[cur],
#             "StoreZIP"        : base_zip[cur],
#             "Cluster"         : node_cluster[cur],
#             "Cube"            : round(demands[cur], 1),
#             "ServiceMin"      : round(svc, 1),
#             "Arrival(DayTime)": fmt_day_time(arrival),
#             "Depart(DayTime)" : fmt_day_time(depart),
#             "DriveHrSoFar"    : round(drive / 60, 2),
#             "DutyHrSoFar"     : round(duty  / 60, 2),
#         })
#         prev = cur

#     # Return to DC
#     ret_t  = travel_min[prev][0]
#     ret_mi = dist_miles[prev][0]

#     rest_inserted = 0.0
#     if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
#         rest_inserted = REST_MIN
#         t += REST_MIN

#     drive_return = ret_t
#     duty_return  = ret_t

#     t += ret_t
#     miles += ret_mi

#     summary_rows.append({
#         "Truck"                  : truck_num,
#         "Stops"                  : stop_n,
#         "TotalCube"              : round(sum(demands[nd] for nd in route if nd != 0), 1),
#         "RouteMiles"             : round(miles, 1),
#         "Dispatch(DayTime)"      : fmt_day_time(dispatch),
#         "ReturnDC(DayTime)"      : fmt_day_time(t),
#         "Dispatch(HH:MM>24)"     : fmt_hhmm(dispatch),
#         "ReturnDC(HH:MM>24)"     : fmt_hhmm(t),
#         "RestInsertedMin"        : int(rest_inserted),
#         "DelivDriveHr"           : round(drive / 60, 2),
#         "DelivDutyHr"            : round(duty  / 60, 2),
#         "ReturnDriveHr"          : round(ret_t / 60, 2),
#         "ReturnDutyHr"           : round(ret_t / 60, 2),
#         "TotalDriveHr"           : round((drive + ret_t) / 60, 2),
#         "TotalDutyHr"            : round((duty  + ret_t)  / 60, 2),
#     })

# schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
# summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

# print("\n--- SUMMARY (first 10 trucks) ---")
# print(summary_df.head(10).to_string(index=False))

# truck_plan_df = schedule_df[[
#     "Truck","Stop#","StoreZIP","Cluster","Arrival(DayTime)","Depart(DayTime)",
#     "Cube","ServiceMin","VisitID",
# ]].copy()
# truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]


# # ==================== 10) SAVE OUTPUT ====================

# out_path = "monday_ortools_routes_schedule.xlsx"
# with pd.ExcelWriter(out_path, engine="openpyxl") as w:
#     summary_df.to_excel(w,    sheet_name="Summary",   index=False)
#     schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
#     truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

# print(f"\nSaved: {out_path}")
# print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
# print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")
# print(f"Cluster mismatch penalty (miles): {CLUSTER_MISMATCH_PENALTY_MILES}")
# ============================================================
# COMPLETE, WORKING END-TO-END CODE (Colab / Python) — MONDAY
# + Soft "cluster grouping" preference using LocationTable!Cluster_Hierarchical
# UPDATED: runtime + complexity proxy
# ============================================================

import glob
import math
import time
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

t_total0 = time.perf_counter()   # total end-to-end start

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200

OPEN       = 8  * 60
CLOSE      = 18 * 60

MAX_DRIVE  = 11 * 60
MAX_DUTY   = 14 * 60
REST_MIN   = 10 * 60

SOLVER_SECONDS = 120
EXTRA_VEHICLES = 30

DAY = 24 * 60

CLUSTER_MISMATCH_PENALTY_MILES = 30.0

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

monday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("monday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("monday",))
)
if monday_sheet is None:
    raise ValueError("Could not find a Monday sheet (e.g., 'Monday Final').")

location_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("location", "table"))
    or pick_sheet(xl.sheet_names, contains_all=("location",))
)
if location_sheet is None:
    raise ValueError("Could not find a LocationTable sheet for clusters.")

print("Distance sheet:", distance_sheet)
print("Monday sheet  :", monday_sheet)
print("Location sheet:", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

t_build0 = time.perf_counter()

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 3.5) READ LOCATION TABLE (ZIP -> Cluster) ====================

loc = pd.read_excel(file_path, sheet_name=location_sheet)
loc.columns = [str(c).strip() for c in loc.columns]

zip_loc_col = next((c for c in ["ZIP", "Zip", "zip", "StoreZIP", "Store Zip"] if c in loc.columns), None)
clu_col = next((c for c in ["Cluster_Hierarchical", "cluster_hierarchical", "Cluster"] if c in loc.columns), None)

if zip_loc_col is None or clu_col is None:
    raise ValueError(f"Location sheet must have ZIP and Cluster_Hierarchical. Found columns: {loc.columns.tolist()}")

loc[zip_loc_col] = loc[zip_loc_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
loc[clu_col] = pd.to_numeric(loc[clu_col], errors="coerce")
loc = loc.dropna(subset=[zip_loc_col, clu_col])

zip_to_cluster = dict(zip(loc[zip_loc_col], loc[clu_col].astype(int)))
print("Clusters loaded for ZIPs:", len(zip_to_cluster))

# ==================== 4) READ & CLEAN MONDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=monday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)

if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Monday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z    = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k         = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk     = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP"        : z,
            "VisitID"        : f"{z}#{part}" if k > 1 else z,
            "Cube"           : chunk,
            "UnloadFromSheet": unload_val,
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

node_cluster = [-1] + [zip_to_cluster.get(z, -1) for z in split_df["BaseZIP"].tolist()]
missing_cluster = sorted({z for z in set(split_df["BaseZIP"]) if z not in zip_to_cluster})
if missing_cluster:
    print("WARNING: These ZIPs have no cluster in LocationTable (treated as -1):", missing_cluster[:20])

n = len(base_zip)

# O(n^2) matrix build
dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

t_build1 = time.perf_counter()

# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

# -------- Complexity proxy (for reporting) --------
# Exact time complexity is not polynomial (VRP is NP-hard).
# Practical work drivers:
#   - Data/matrix prep: O(n^2)
#   - Heuristic search: bounded by SOLVER_SECONDS (time limit)
#   - Route extraction/scheduling: ~O(total stops)
print("\n--- Complexity proxy ---")
print(f"n (nodes incl. DC) = {n}")
print(f"V (vehicles avail) = {num_vehicles}")
print(f"Matrix build       ~ O(n^2) = {n*n:,} pair evaluations")
print(f"Solver             ~ time-bounded metaheuristic (<= {SOLVER_SECONDS}s)")
print("------------------------\n")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

cluster_penalty_cost = int(CLUSTER_MISMATCH_PENALTY_MILES * 1000)

def arc_cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)

    base_cost = int(dist_miles[i][j] * 1000)

    if i != 0 and j != 0:
        ci = node_cluster[i]
        cj = node_cluster[j]
        if ci != -1 and cj != -1 and ci != cj:
            base_cost += cluster_penalty_cost

    return base_cost

routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(arc_cost_cb))

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0, [CAPACITY] * num_vehicles, True, "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(
            f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} min too large for 08–18."
        )
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.FromSeconds(SOLVER_SECONDS)

t_solve0 = time.perf_counter()
solution = routing.SolveWithParameters(params)
t_solve1 = time.perf_counter()

print("\n--- Runtime ---")
print(f"Build/prep time      : {(t_build1 - t_build0):.3f} s")
print(f"OR-Tools solve time  : {(t_solve1 - t_solve0):.3f} s (limit={SOLVER_SECONDS}s)")
print("----------------\n")

if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "All stops cannot be served 08:00–18:00 with given capacity.\n"
        "Try increasing EXTRA_VEHICLES or check input data."
    )

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck"           : truck_num,
            "Stop#"           : stop_n,
            "VisitID"         : nodes_id[cur],
            "StoreZIP"        : base_zip[cur],
            "Cluster"         : node_cluster[cur],
            "Cube"            : round(demands[cur], 1),
            "ServiceMin"      : round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar"    : round(drive / 60, 2),
            "DutyHrSoFar"     : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck"              : truck_num,
        "Stops"              : stop_n,
        "TotalCube"          : round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles"         : round(miles, 1),
        "Dispatch(DayTime)"  : fmt_day_time(dispatch),
        "ReturnDC(DayTime)"  : fmt_day_time(t),
        "Dispatch(HH:MM>24)" : fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)" : fmt_hhmm(t),
        "RestInsertedMin"    : int(rest_inserted),
        "TotalDriveHr"       : round((drive + ret_t) / 60, 2),
        "TotalDutyHr"        : round((duty  + ret_t)  / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Cluster","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID",
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = "monday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

t_total1 = time.perf_counter()

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")
print(f"Cluster mismatch penalty (miles): {CLUSTER_MISMATCH_PENALTY_MILES}")

print("\n--- Total runtime ---")
print(f"End-to-end runtime : {(t_total1 - t_total0):.3f} s")
print("---------------------")

Using file: solution deliveries Monday (1).xlsx
Sheets: ['Sheet1', 'OrderTable', 'monday pivot', 'Monday Final', 'Monday', 'distance', 'LocationTable']
Distance sheet: distance
Monday sheet  : Monday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Clusters loaded for ZIPs: 123
Monday stores (original rows): 34
Total cube: 10223.0
Total delivery stops after splitting: 34
Min trailers (capacity only): 4
Vehicles given to OR-Tools  : 34

--- Complexity proxy ---
n (nodes incl. DC) = 35
V (vehicles avail) = 34
Matrix build       ~ O(n^2) = 1,225 pair evaluations
Solver             ~ time-bounded metaheuristic (<= 120s)
------------------------


--- Runtime ---
Build/prep time      : 1.964 s
OR-Tools solve time  : 120.001 s (limit=120s)
----------------

Routes from OR-Tools (before HOS split): 5
Routes after HOS enforcement: 5

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  TotalDriveHr  TotalDutyHr
     1     11     3179.0       413.0          D0 04:21          D1 06:25              04:21              30:25              600         10.32        16.06
     2      9     1738.0        73.0          D0 07:48          D0 

Tuesday

In [ ]:
# ============================================================
# COMPLETE, WORKING END-TO-END CODE (Colab / Python) — TUESDAY
# + Soft "cluster grouping" preference using LocationTable!Cluster_Hierarchical
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200         # cubic feet per trailer

OPEN       = 8  * 60     # 08:00 in minutes
CLOSE      = 18 * 60     # 18:00 in minutes

# HOS limits (minutes)
MAX_DRIVE  = 11 * 60     # 660 min — cumulative DRIVING only
MAX_DUTY   = 14 * 60     # 840 min — cumulative ON-DUTY (drive + service + wait)
REST_MIN   = 10 * 60     # 600 min — mandatory rest before reset

SOLVER_SECONDS = 120     # OR-Tools time limit
EXTRA_VEHICLES = 30      # padding so solver has enough vehicles

DAY = 24 * 60            # minutes in a day (for display)

# ---------- Cluster preference strength ----------
# Penalty added (in "miles") when consecutive stops are in different clusters.
# Larger => stronger grouping, but too large may increase total miles a lot.
CLUSTER_MISMATCH_PENALTY_MILES = 30.0  # try 10, 20, 30, 50 if needed


# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"


# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)


# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

tuesday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("tuesday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("tuesday",))
)
if tuesday_sheet is None:
    raise ValueError("Could not find a Tuesday sheet (e.g., 'Tuesday Final').")

# Location table sheet (where clusters live)
location_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("location", "table"))
    or pick_sheet(xl.sheet_names, contains_all=("location",))
)
if location_sheet is None:
    raise ValueError("Could not find a LocationTable sheet for clusters.")

print("Distance sheet:", distance_sheet)
print("Tuesday sheet :", tuesday_sheet)
print("Location sheet:", location_sheet)


# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")


# ==================== 3.5) READ LOCATION TABLE (ZIP -> Cluster) ====================

loc = pd.read_excel(file_path, sheet_name=location_sheet)
loc.columns = [str(c).strip() for c in loc.columns]

zip_loc_col = next((c for c in ["ZIP", "Zip", "zip", "StoreZIP", "Store Zip"] if c in loc.columns), None)
clu_col = next((c for c in ["Cluster_Hierarchical", "cluster_hierarchical", "Cluster"] if c in loc.columns), None)

if zip_loc_col is None or clu_col is None:
    raise ValueError(f"Location sheet must have ZIP and Cluster_Hierarchical. Found columns: {loc.columns.tolist()}")

loc[zip_loc_col] = loc[zip_loc_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
loc[clu_col] = pd.to_numeric(loc[clu_col], errors="coerce")
loc = loc.dropna(subset=[zip_loc_col, clu_col])

zip_to_cluster = dict(zip(loc[zip_loc_col], loc[clu_col].astype(int)))
print("Clusters loaded for ZIPs:", len(zip_to_cluster))


# ==================== 4) READ & CLEAN TUESDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=tuesday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)

if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Tuesday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")


# ==================== 4.1) OPTIONAL: DROP DC->DC ROWS IF PRESENT ====================
# If Tuesday sheet contains a "01887 -> 01887" (DC to DC) row, remove it.
day_df = day_df[day_df[zip_col] != DC].copy()


# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z    = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k         = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk     = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP"        : z,
            "VisitID"        : f"{z}#{part}" if k > 1 else z,
            "Cube"           : chunk,
            "UnloadFromSheet": unload_val,
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")


# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()   # index 0 = depot
base_zip = [DC] + split_df["BaseZIP"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

# Cluster per node (depot cluster = -1)
node_cluster = [-1] + [zip_to_cluster.get(z, -1) for z in split_df["BaseZIP"].tolist()]
missing_cluster = sorted({z for z in set(split_df["BaseZIP"]) if z not in zip_to_cluster})
if missing_cluster:
    print("WARNING: These ZIPs have no cluster in LocationTable (treated as -1):", missing_cluster[:20])

n = len(base_zip)

# Distance matrix (miles)
dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]

# Travel time (minutes): 40 mph → 1 mile = 1.5 min
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

# Demand (cube) per node; depot = 0
demands = [0] + split_df["Cube"].astype(float).tolist()

# Service time per node
service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)


# ==================== 7) OR-TOOLS VRP (capacity + time window + cluster preference) ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

# --- ARC COST = distance + penalty(if cluster mismatch) ---
cluster_penalty_cost = int(CLUSTER_MISMATCH_PENALTY_MILES * 1000)

def arc_cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)

    base_cost = int(dist_miles[i][j] * 1000)

    if i != 0 and j != 0:
        ci = node_cluster[i]
        cj = node_cluster[j]
        if ci != -1 and cj != -1 and ci != cj:
            base_cost += cluster_penalty_cost

    return base_cost

routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(arc_cost_cb))

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0, [CAPACITY] * num_vehicles, True, "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(
            f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} min too large for 08–18."
        )
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "All stops cannot be served 08:00–18:00 with given capacity.\n"
        "Try increasing EXTRA_VEHICLES or check input data."
    )

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")


# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")


# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck"           : truck_num,
            "Stop#"           : stop_n,
            "VisitID"         : nodes_id[cur],
            "StoreZIP"        : base_zip[cur],
            "Cluster"         : node_cluster[cur],
            "Cube"            : round(demands[cur], 1),
            "ServiceMin"      : round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar"    : round(drive / 60, 2),
            "DutyHrSoFar"     : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck"                  : truck_num,
        "Stops"                  : stop_n,
        "TotalCube"              : round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles"             : round(miles, 1),
        "Dispatch(DayTime)"      : fmt_day_time(dispatch),
        "ReturnDC(DayTime)"      : fmt_day_time(t),
        "Dispatch(HH:MM>24)"     : fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)"     : fmt_hhmm(t),
        "RestInsertedMin"        : int(rest_inserted),
        "DelivDriveHr"           : round(drive / 60, 2),
        "DelivDutyHr"            : round(duty  / 60, 2),
        "TotalDriveHr"           : round((drive + ret_t) / 60, 2),
        "TotalDutyHr"            : round((duty  + ret_t)  / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Cluster","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID",
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]


# ==================== 10) SAVE OUTPUT ====================

out_path = "tuesday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")
print(f"Cluster mismatch penalty (miles): {CLUSTER_MISMATCH_PENALTY_MILES}")


Using file: solution deliveries Tuesday.xlsx
Sheets: ['Sheet1', 'Sheet3', 'OrderTable', 'Tuesday Final', 'Tues', 'distance', 'LocationTable']
Distance sheet: distance
Tuesday sheet : Tuesday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Clusters loaded for ZIPs: 123
Tuesday stores (original rows): 41
Total cube: 11537.0
Total delivery stops after splitting: 40
Min trailers (capacity only): 4
Vehicles given to OR-Tools  : 34


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 5
Routes after HOS enforcement: 5

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  DelivDriveHr  DelivDutyHr  TotalDriveHr  TotalDutyHr
     1     10     3040.0       456.0          D0 03:21          D1 05:45              03:21              29:45              600          8.38        13.38         11.40        16.40
     2      4      809.0       447.0          D0 05:08          D1 04:18              05:08              28:18              600          9.75        11.75         11.18        13.18
     3     10     2337.0       308.0          D0 05:15          D0 17:57              05:15              17:57                0          4.95         9.95          7.70        12.70
     4      7     2200.0        80.0          D0 07:34          D0 13:04              07:34              13:04                0          1.73         5.22        

Wednesday

In [ ]:
# ============================================================
# COMPLETE, WORKING END-TO-END CODE (Colab / Python) — WEDNESDAY
# + Soft "cluster grouping" preference using LocationTable!Cluster_Hierarchical
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200         # cubic feet per trailer

OPEN       = 8  * 60     # 08:00 in minutes
CLOSE      = 18 * 60     # 18:00 in minutes

# HOS limits (minutes)
MAX_DRIVE  = 11 * 60     # 660 min — cumulative DRIVING only
MAX_DUTY   = 14 * 60     # 840 min — cumulative ON-DUTY (drive + service + wait)
REST_MIN   = 10 * 60     # 600 min — mandatory rest before reset

SOLVER_SECONDS = 120     # OR-Tools time limit
EXTRA_VEHICLES = 30      # padding so solver has enough vehicles

DAY = 24 * 60            # minutes in a day (for display)

# ---------- Cluster preference strength ----------
CLUSTER_MISMATCH_PENALTY_MILES = 30.0  # try 10, 20, 30, 50 if needed


# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"


# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)


# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

wednesday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("wednesday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("wednesday",))
)
if wednesday_sheet is None:
    raise ValueError("Could not find a Wednesday sheet (e.g., 'Wednesday Final').")

# Location table sheet (where clusters live)
location_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("location", "table"))
    or pick_sheet(xl.sheet_names, contains_all=("location",))
)
if location_sheet is None:
    raise ValueError("Could not find a LocationTable sheet for clusters.")

print("Distance sheet :", distance_sheet)
print("Wednesday sheet:", wednesday_sheet)
print("Location sheet :", location_sheet)


# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")


# ==================== 3.5) READ LOCATION TABLE (ZIP -> Cluster) ====================

loc = pd.read_excel(file_path, sheet_name=location_sheet)
loc.columns = [str(c).strip() for c in loc.columns]

zip_loc_col = next((c for c in ["ZIP", "Zip", "zip", "StoreZIP", "Store Zip"] if c in loc.columns), None)
clu_col = next((c for c in ["Cluster_Hierarchical", "cluster_hierarchical", "Cluster"] if c in loc.columns), None)

if zip_loc_col is None or clu_col is None:
    raise ValueError(f"Location sheet must have ZIP and Cluster_Hierarchical. Found columns: {loc.columns.tolist()}")

loc[zip_loc_col] = loc[zip_loc_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
loc[clu_col] = pd.to_numeric(loc[clu_col], errors="coerce")
loc = loc.dropna(subset=[zip_loc_col, clu_col])

zip_to_cluster = dict(zip(loc[zip_loc_col], loc[clu_col].astype(int)))
print("Clusters loaded for ZIPs:", len(zip_to_cluster))


# ==================== 4) READ & CLEAN WEDNESDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=wednesday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)

if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Wednesday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# Drop DC->DC if present
day_df = day_df[day_df[zip_col] != DC].copy()


# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z    = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k         = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk     = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP"        : z,
            "VisitID"        : f"{z}#{part}" if k > 1 else z,
            "Cube"           : chunk,
            "UnloadFromSheet": unload_val,
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")


# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

node_cluster = [-1] + [zip_to_cluster.get(z, -1) for z in split_df["BaseZIP"].tolist()]
missing_cluster = sorted({z for z in set(split_df["BaseZIP"]) if z not in zip_to_cluster})
if missing_cluster:
    print("WARNING: These ZIPs have no cluster in LocationTable (treated as -1):", missing_cluster[:20])

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)


# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

cluster_penalty_cost = int(CLUSTER_MISMATCH_PENALTY_MILES * 1000)

def arc_cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    base_cost = int(dist_miles[i][j] * 1000)

    if i != 0 and j != 0:
        ci = node_cluster[i]
        cj = node_cluster[j]
        if ci != -1 and cj != -1 and ci != cj:
            base_cost += cluster_penalty_cost
    return base_cost

routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(arc_cost_cb))

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0, [CAPACITY] * num_vehicles, True, "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} min too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError("OR-Tools found no feasible solution. Check data / constraints.")

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")


# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")


# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck"           : truck_num,
            "Stop#"           : stop_n,
            "VisitID"         : nodes_id[cur],
            "StoreZIP"        : base_zip[cur],
            "Cluster"         : node_cluster[cur],
            "Cube"            : round(demands[cur], 1),
            "ServiceMin"      : round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar"    : round(drive / 60, 2),
            "DutyHrSoFar"     : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck"                  : truck_num,
        "Stops"                  : stop_n,
        "TotalCube"              : round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles"             : round(miles, 1),
        "Dispatch(DayTime)"      : fmt_day_time(dispatch),
        "ReturnDC(DayTime)"      : fmt_day_time(t),
        "Dispatch(HH:MM>24)"     : fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)"     : fmt_hhmm(t),
        "RestInsertedMin"        : int(rest_inserted),
        "DelivDriveHr"           : round(drive / 60, 2),
        "DelivDutyHr"            : round(duty  / 60, 2),
        "TotalDriveHr"           : round((drive + ret_t) / 60, 2),
        "TotalDutyHr"            : round((duty  + ret_t)  / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Cluster","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID",
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]


# ==================== 10) SAVE OUTPUT ====================

out_path = "wednesday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")
print(f"Cluster mismatch penalty (miles): {CLUSTER_MISMATCH_PENALTY_MILES}")


Using file: solution deliveries Wednesday.xlsx
Sheets: ['Sheet1', 'Sheet2', 'OrderTable', 'Wednesday Final', 'distance', 'LocationTable']
Distance sheet : distance
Wednesday sheet: Wednesday Final
Location sheet : LocationTable
Distance matrix shape: (123, 123)
Clusters loaded for ZIPs: 123
Wednesday stores (original rows): 39
Total cube: 15192.0
Total delivery stops after splitting: 39
Min trailers (capacity only): 5
Vehicles given to OR-Tools  : 35


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 6
Routes after HOS enforcement: 6

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  DelivDriveHr  DelivDutyHr  TotalDriveHr  TotalDutyHr
     1      5     3172.0       220.0          D0 05:48          D0 14:24              05:48              14:24                0          3.75         6.85          5.50         8.60
     2      9     1605.0       323.0          D0 07:14          D0 19:48              07:14              19:48                0          5.90        10.40          8.07        12.57
     3      2     2959.0        36.0          D0 07:28          D0 10:16              07:28              10:16                0          0.75         2.65          0.90         2.80
     4      5     2858.0       107.0          D0 07:34          D0 13:17              07:34              13:17                0          2.20         5.24        

Thursday

In [ ]:
# ============================================================
# COMPLETE, WORKING END-TO-END CODE (Colab / Python) — THURSDAY
# + Soft "cluster grouping" preference using LocationTable!Cluster_Hierarchical
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200         # cubic feet per trailer

OPEN       = 8  * 60     # 08:00 in minutes
CLOSE      = 18 * 60     # 18:00 in minutes

# HOS limits (minutes)
MAX_DRIVE  = 11 * 60     # 660 min — cumulative DRIVING only
MAX_DUTY   = 14 * 60     # 840 min — cumulative ON-DUTY (drive + service + wait)
REST_MIN   = 10 * 60     # 600 min — mandatory rest before reset

SOLVER_SECONDS = 120     # OR-Tools time limit
EXTRA_VEHICLES = 30      # padding so solver has enough vehicles

DAY = 24 * 60            # minutes in a day (for display)

# ---------- Cluster preference strength ----------
CLUSTER_MISMATCH_PENALTY_MILES = 30.0  # try 10, 20, 30, 50 if needed


# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"


# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)


# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

thursday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("thursday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("thursday",))
    or pick_sheet(xl.sheet_names, contains_all=("thur", "final"))  # sometimes abbreviated
    or pick_sheet(xl.sheet_names, contains_all=("thur",))
)
if thursday_sheet is None:
    raise ValueError("Could not find a Thursday sheet (e.g., 'Thursday Final').")

# Location table sheet (where clusters live)
location_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("location", "table"))
    or pick_sheet(xl.sheet_names, contains_all=("location",))
)
if location_sheet is None:
    raise ValueError("Could not find a LocationTable sheet for clusters.")

print("Distance sheet:", distance_sheet)
print("Thursday sheet:", thursday_sheet)
print("Location sheet:", location_sheet)


# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")


# ==================== 3.5) READ LOCATION TABLE (ZIP -> Cluster) ====================

loc = pd.read_excel(file_path, sheet_name=location_sheet)
loc.columns = [str(c).strip() for c in loc.columns]

zip_loc_col = next((c for c in ["ZIP", "Zip", "zip", "StoreZIP", "Store Zip"] if c in loc.columns), None)
clu_col = next((c for c in ["Cluster_Hierarchical", "cluster_hierarchical", "Cluster"] if c in loc.columns), None)

if zip_loc_col is None or clu_col is None:
    raise ValueError(f"Location sheet must have ZIP and Cluster_Hierarchical. Found columns: {loc.columns.tolist()}")

loc[zip_loc_col] = loc[zip_loc_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
loc[clu_col] = pd.to_numeric(loc[clu_col], errors="coerce")
loc = loc.dropna(subset=[zip_loc_col, clu_col])

zip_to_cluster = dict(zip(loc[zip_loc_col], loc[clu_col].astype(int)))
print("Clusters loaded for ZIPs:", len(zip_to_cluster))


# ==================== 4) READ & CLEAN THURSDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=thursday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)

if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Thursday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# Drop DC->DC if present
day_df = day_df[day_df[zip_col] != DC].copy()


# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z    = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k         = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk     = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP"        : z,
            "VisitID"        : f"{z}#{part}" if k > 1 else z,
            "Cube"           : chunk,
            "UnloadFromSheet": unload_val,
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")


# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

node_cluster = [-1] + [zip_to_cluster.get(z, -1) for z in split_df["BaseZIP"].tolist()]
missing_cluster = sorted({z for z in set(split_df["BaseZIP"]) if z not in zip_to_cluster})
if missing_cluster:
    print("WARNING: These ZIPs have no cluster in LocationTable (treated as -1):", missing_cluster[:20])

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)


# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

cluster_penalty_cost = int(CLUSTER_MISMATCH_PENALTY_MILES * 1000)

def arc_cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    base_cost = int(dist_miles[i][j] * 1000)

    if i != 0 and j != 0:
        ci = node_cluster[i]
        cj = node_cluster[j]
        if ci != -1 and cj != -1 and ci != cj:
            base_cost += cluster_penalty_cost
    return base_cost

routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(arc_cost_cb))

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0, [CAPACITY] * num_vehicles, True, "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} min too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError("OR-Tools found no feasible solution. Check data / constraints.")

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")


# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")


# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck"           : truck_num,
            "Stop#"           : stop_n,
            "VisitID"         : nodes_id[cur],
            "StoreZIP"        : base_zip[cur],
            "Cluster"         : node_cluster[cur],
            "Cube"            : round(demands[cur], 1),
            "ServiceMin"      : round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar"    : round(drive / 60, 2),
            "DutyHrSoFar"     : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck"                  : truck_num,
        "Stops"                  : stop_n,
        "TotalCube"              : round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles"             : round(miles, 1),
        "Dispatch(DayTime)"      : fmt_day_time(dispatch),
        "ReturnDC(DayTime)"      : fmt_day_time(t),
        "Dispatch(HH:MM>24)"     : fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)"     : fmt_hhmm(t),
        "RestInsertedMin"        : int(rest_inserted),
        "DelivDriveHr"           : round(drive / 60, 2),
        "DelivDutyHr"            : round(duty  / 60, 2),
        "TotalDriveHr"           : round((drive + ret_t) / 60, 2),
        "TotalDutyHr"            : round((duty  + ret_t)  / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Cluster","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID",
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]


# ==================== 10) SAVE OUTPUT ====================

out_path = "thursday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")
print(f"Cluster mismatch penalty (miles): {CLUSTER_MISMATCH_PENALTY_MILES}")


Using file: solution deliveries Thursday.xlsx
Sheets: ['Sheet1', 'Sheet2', 'OrderTable', 'Thursday Final', 'distance', 'LocationTable']
Distance sheet: distance
Thursday sheet: Thursday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Clusters loaded for ZIPs: 123
Thursday stores (original rows): 45
Total cube: 15009.0
Total delivery stops after splitting: 45
Min trailers (capacity only): 5
Vehicles given to OR-Tools  : 35


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 6
Routes after HOS enforcement: 6

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  DelivDriveHr  DelivDutyHr  TotalDriveHr  TotalDutyHr
     1      8     1938.0       434.0          D0 05:30          D1 06:21              05:30              30:21              600          7.97        11.97         10.85        14.85
     2     10     1988.0       412.0          D0 04:21          D1 05:39              04:21              29:39              600          7.03        12.03         10.30        15.30
     3      7     1826.0       344.0          D0 06:36          D0 18:42              06:36              18:42                0          5.80         9.30          8.60        12.10
     4      2     3196.0        45.0          D0 07:32          D0 10:27              07:32              10:27                0          0.57         2.38        

Friday

In [ ]:
# ============================================================
# COMPLETE, WORKING END-TO-END CODE (Colab / Python) — FRIDAY
# + Soft "cluster grouping" preference using LocationTable!Cluster_Hierarchical
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200         # cubic feet per trailer

OPEN       = 8  * 60     # 08:00 in minutes
CLOSE      = 18 * 60     # 18:00 in minutes

# HOS limits (minutes)
MAX_DRIVE  = 11 * 60     # 660 min — cumulative DRIVING only
MAX_DUTY   = 14 * 60     # 840 min — cumulative ON-DUTY (drive + service + wait)
REST_MIN   = 10 * 60     # 600 min — mandatory rest before reset

SOLVER_SECONDS = 120     # OR-Tools time limit
EXTRA_VEHICLES = 30      # padding so solver has enough vehicles

DAY = 24 * 60            # minutes in a day (for display)

# ---------- Cluster preference strength ----------
CLUSTER_MISMATCH_PENALTY_MILES = 30.0  # try 10, 20, 30, 50 if needed


# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"


# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)


# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

friday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("friday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("friday",))
)
if friday_sheet is None:
    raise ValueError("Could not find a Friday sheet (e.g., 'Friday Final').")

# Location table sheet (where clusters live)
location_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("location", "table"))
    or pick_sheet(xl.sheet_names, contains_all=("location",))
)
if location_sheet is None:
    raise ValueError("Could not find a LocationTable sheet for clusters.")

print("Distance sheet:", distance_sheet)
print("Friday sheet  :", friday_sheet)
print("Location sheet:", location_sheet)


# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")


# ==================== 3.5) READ LOCATION TABLE (ZIP -> Cluster) ====================

loc = pd.read_excel(file_path, sheet_name=location_sheet)
loc.columns = [str(c).strip() for c in loc.columns]

zip_loc_col = next((c for c in ["ZIP", "Zip", "zip", "StoreZIP", "Store Zip"] if c in loc.columns), None)
clu_col = next((c for c in ["Cluster_Hierarchical", "cluster_hierarchical", "Cluster"] if c in loc.columns), None)

if zip_loc_col is None or clu_col is None:
    raise ValueError(f"Location sheet must have ZIP and Cluster_Hierarchical. Found columns: {loc.columns.tolist()}")

loc[zip_loc_col] = loc[zip_loc_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
loc[clu_col] = pd.to_numeric(loc[clu_col], errors="coerce")
loc = loc.dropna(subset=[zip_loc_col, clu_col])

zip_to_cluster = dict(zip(loc[zip_loc_col], loc[clu_col].astype(int)))
print("Clusters loaded for ZIPs:", len(zip_to_cluster))


# ==================== 4) READ & CLEAN FRIDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=friday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)

if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Friday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# Drop DC->DC if present
day_df = day_df[day_df[zip_col] != DC].copy()


# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z    = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k         = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk     = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP"        : z,
            "VisitID"        : f"{z}#{part}" if k > 1 else z,
            "Cube"           : chunk,
            "UnloadFromSheet": unload_val,
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")


# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

node_cluster = [-1] + [zip_to_cluster.get(z, -1) for z in split_df["BaseZIP"].tolist()]
missing_cluster = sorted({z for z in set(split_df["BaseZIP"]) if z not in zip_to_cluster})
if missing_cluster:
    print("WARNING: These ZIPs have no cluster in LocationTable (treated as -1):", missing_cluster[:20])

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)


# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

cluster_penalty_cost = int(CLUSTER_MISMATCH_PENALTY_MILES * 1000)

def arc_cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    base_cost = int(dist_miles[i][j] * 1000)

    if i != 0 and j != 0:
        ci = node_cluster[i]
        cj = node_cluster[j]
        if ci != -1 and cj != -1 and ci != cj:
            base_cost += cluster_penalty_cost
    return base_cost

routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(arc_cost_cb))

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0, [CAPACITY] * num_vehicles, True, "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} min too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError("OR-Tools found no feasible solution. Check data / constraints.")

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")


# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")


# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck"           : truck_num,
            "Stop#"           : stop_n,
            "VisitID"         : nodes_id[cur],
            "StoreZIP"        : base_zip[cur],
            "Cluster"         : node_cluster[cur],
            "Cube"            : round(demands[cur], 1),
            "ServiceMin"      : round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar"    : round(drive / 60, 2),
            "DutyHrSoFar"     : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck"                  : truck_num,
        "Stops"                  : stop_n,
        "TotalCube"              : round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles"             : round(miles, 1),
        "Dispatch(DayTime)"      : fmt_day_time(dispatch),
        "ReturnDC(DayTime)"      : fmt_day_time(t),
        "Dispatch(HH:MM>24)"     : fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)"     : fmt_hhmm(t),
        "RestInsertedMin"        : int(rest_inserted),
        "DelivDriveHr"           : round(drive / 60, 2),
        "DelivDutyHr"            : round(duty  / 60, 2),
        "TotalDriveHr"           : round((drive + ret_t) / 60, 2),
        "TotalDutyHr"            : round((duty  + ret_t)  / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Cluster","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID",
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]


# ==================== 10) SAVE OUTPUT ====================

out_path = "friday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")
print(f"Cluster mismatch penalty (miles): {CLUSTER_MISMATCH_PENALTY_MILES}")


Using file: solution deliveries Friday.xlsx
Sheets: ['Sheet1', 'Sheet2', 'OrderTable', 'Friday Final', 'distance', 'LocationTable']
Distance sheet: distance
Friday sheet  : Friday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Clusters loaded for ZIPs: 123
Friday stores (original rows): 41
Total cube: 13468.0
Total delivery stops after splitting: 42
Min trailers (capacity only): 5
Vehicles given to OR-Tools  : 35


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 6
Routes after HOS enforcement: 6

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  DelivDriveHr  DelivDutyHr  TotalDriveHr  TotalDutyHr
     1     12     2636.0       310.0          D0 07:14          D0 20:58              07:14              20:58                0          4.62        10.62          7.75        13.75
     2     10     1649.0        75.0          D0 07:34          D0 14:27              07:34              14:27                0          1.73         6.72          1.88         6.88
     3      1      177.0       198.0          D0 05:32          D0 10:58              05:32              10:58                0          2.48         2.98          4.95         5.45
     4      7     2687.0       126.0          D0 07:44          D0 15:31              07:44              15:31                0          2.95         7.59        